# YOLOv8 cup detection fine-tuning

## Environment Setup and Imports

This cell initializes the environment required for training and evaluating a YOLOv8 object detection model.

We  define the root directory of the project and locate the dataset configuration file (`cup_small.yaml`). This configuration file specifies the dataset structure, including the paths to training and validation images and the list of class names.

Finally, we print the resolved paths and verify that the dataset configuration file exists. This serves as a sanity check to ensure that the project structure is correct before proceeding.


In [2]:
# Environment setup & imports

import os
from pathlib import Path

from ultralytics import YOLO

PROJECT_ROOT = Path("../../smart-object-detection").resolve()
DATA_CONFIG = PROJECT_ROOT / "configs" / "data" / "cup_small.yaml"

print("Project root:", PROJECT_ROOT)
print("Data config:", DATA_CONFIG)
print("Exists?", DATA_CONFIG.exists())


Project root: D:\an4sem1\PRS\smart-object-detection
Data config: D:\an4sem1\PRS\smart-object-detection\configs\data\cup_small.yaml
Exists? True


## Dataset Configuration Check

In this cell, we load and display the contents of the dataset configuration file (`cup_small.yaml`).

The file is written in YAML format and defines:
- the locations of the training and validation datasets
- the number of object classes

Printing the configuration allows us to confirm that the dataset is correctly defined and that the class labels match the annotations used during training.


In [3]:
# Show cup.yaml content

import yaml

with open(DATA_CONFIG, "r") as f:
    cfg = yaml.safe_load(f)

print(cfg)


{'path': 'D:/an4sem1/PRS/smart-object-detection/cup_coco_yolo_small', 'train': 'images/train2017', 'val': 'images/val2017', 'nc': 1, 'names': {0: 'cup'}}


## Training Parameters

This cell defines the main training hyperparameters for the YOLOv8 model.

- `MODEL_NAME` specifies the pretrained YOLOv8 model used as a starting point.
- `EPOCHS` defines the maximum number of training passes over the dataset.
- `IMG_SIZE` sets the image resolution used during training.
- `BATCH_SIZE` controls how many images are processed simultaneously.
- `EXPERIMENT_NAME` is used to organize outputs from this experiment.

An output directory is also created to store training logs, model weights, and evaluation results. The directory is created safely using `parents=True` and `exist_ok=True`, allowing the cell to be re-run without errors.


In [4]:
# Define training parameters

MODEL_NAME = "yolov8n.pt"
EPOCHS = 25
IMG_SIZE = 512
BATCH_SIZE = 8
EXPERIMENT_NAME = "cup_yolov8n_small_cpu"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output dir:", OUTPUT_DIR)


Output dir: D:\an4sem1\PRS\smart-object-detection\outputs


## YOLOv8 Model Training

In this cell, we initialize the YOLOv8 model using pretrained weights and fine-tune it on the custom cup detection dataset.

The training process uses:
- the dataset configuration file
- a fixed image size and batch size
- CPU-based training due to hardware constraints
- disk caching to speed up data loading

During training, the model learns to adapt its pretrained features to the specific task of detecting cups.


In [16]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_CONFIG),
    epochs=20,
    imgsz=512,
    batch=8,
    device="cpu",
    cache="disk",
    project=str(OUTPUT_DIR),
    name="cup_yolov8n_small_cpu",
    exist_ok=True,
)


New https://pypi.org/project/ultralytics/8.4.6 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.12.7 torch-2.9.1+cpu CPU (12th Gen Intel Core(TM) i5-12450H)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=D:\an4sem1\PRS\smart-object-detection\configs\data\cup_small.yaml, epochs=20, time=None, patience=100, batch=8, imgsz=512, save=True, save_period=-1, cache=disk, device=cpu, workers=0, project=D:\an4sem1\PRS\smart-object-detection\outputs, name=cup_yolov8n_small_cpu, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=

train: Scanning D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\labels\train2017... 6000 images, 3000 backgrounds, 0 corrupt: 100%|██████████| 6000/6


train: New cache created: D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\labels\train2017.cache


train: Caching images (4.7GB Disk): 100%|██████████| 6000/6000 [00:41<00:00, 144.62it/s]
D:\an4sem1\PRS\smart-object-detection\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\labels\val2017... 780 images, 390 backgrounds, 0 corrupt: 100%|██████████| 780/780 [00:


val: New cache created: D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\labels\val2017.cache


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      20/20         0G      1.227      1.485      1.107          6        512: 100%|██████████| 750/750 [1:14:57<00:00,  6.00s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 49/49 [02:54<00:00,  3.57s/it]


                   all        780        899      0.546      0.423      0.454      0.316

20 epochs completed in 29.884 hours.
Optimizer stripped from D:\an4sem1\PRS\smart-object-detection\outputs\cup_yolov8n_small_cpu\weights\last.pt, 5.6MB
Optimizer stripped from D:\an4sem1\PRS\smart-object-detection\outputs\cup_yolov8n_small_cpu\weights\best.pt, 5.6MB

Validating D:\an4sem1\PRS\smart-object-detection\outputs\cup_yolov8n_small_cpu\weights\best.pt...
Ultralytics 8.3.0  Python-3.12.7 torch-2.9.1+cpu CPU (12th Gen Intel Core(TM) i5-12450H)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients, 6.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 49/49 [02:33<00:00,  3.13s/it]


                   all        780        899      0.546      0.423      0.455      0.316
Speed: 1.9ms preprocess, 172.0ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to D:\an4sem1\PRS\smart-object-detection\outputs\cup_yolov8n_small_cpu


## Model Validation and Quantitative Evaluation

After training, this cell reloads the best-performing model checkpoint based on validation performance.

The model is evaluated on the validation set using a resolution and custom confidence and IoU thresholds. This allows us to analyze the trade-off between precision and recall.

Key detection metrics are extracted and printed:
- Precision
- Recall
- mAP@0.50
- mAP@0.50:0.95

These metrics summarize both detection accuracy and localization quality and are used for comparison with other object detection models.


In [10]:
# Re-evaluate tuned model

best_pt = OUTPUT_DIR / "cup_yolov8n_small_cpu" / "weights" / "best.pt"
model = YOLO(str(best_pt))

val_results = model.val(
    data=str(DATA_CONFIG),
    imgsz=960,
    conf=0.10,
    iou=0.60,
    save=True
)

box = val_results.box

print("Validation metrics (cup detection):")
print(f"Precision:      {box.mp:.4f}")
print(f"Recall:         {box.mr:.4f}")
print(f"mAP@0.50:       {box.map50:.4f}")
print(f"mAP@0.50:0.95:  {box.map:.4f}")


Ultralytics 8.3.0  Python-3.12.7 torch-2.9.1+cpu CPU (12th Gen Intel Core(TM) i5-12450H)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients, 6.8 GFLOPs


val: Scanning D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\labels\val2017.cache... 780 images, 390 backgrounds, 0 corrupt: 100%|██████████| 780/780 [00:00<?, ?it/s]
D:\an4sem1\PRS\smart-object-detection\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 49/49 [01:46<00:00,  2.18s/it]


                   all        780        899      0.599      0.474      0.532      0.389
Speed: 1.2ms preprocess, 124.5ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to D:\an4sem1\PRS\smart-object-detection\runs\detect\val6
Validation metrics (cup detection):
Precision:      0.5990
Recall:         0.4739
mAP@0.50:       0.5321
mAP@0.50:0.95:  0.3888


## Qualitative Evaluation on Validation Images

This cell performs inference on the validation image set and saves visual prediction results.

By applying the trained model to unseen validation images, we can visually inspect:
- correct detections
- missed objects
- false positives
- bounding box quality

Visual inspection complements quantitative metrics and provides insight into how the model behaves in practice. The annotated images are saved to the output directory for further analysis.


In [7]:
# Test prediction on a few validation images and visualize

VAL_IMAGES_DIR = PROJECT_ROOT / "cup_coco_yolo_small" / "images" / "val2017"
print("Val images dir:", VAL_IMAGES_DIR, " | Exists:", VAL_IMAGES_DIR.exists())

sample_source = str(VAL_IMAGES_DIR)

pred_results = model.predict(
    source=sample_source,
    imgsz=960,
    conf = 0.15,
    iou = 0.55,
    save=True,
    project=str(OUTPUT_DIR),
    name=f"{EXPERIMENT_NAME}_val_pred",
    exist_ok=True,
)

pred_results[:3]


Val images dir: D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\images\val2017  | Exists: True

image 1/780 D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\images\val2017\000000000872.jpg: 960x960 (no detections), 561.9ms
image 2/780 D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\images\val2017\000000002006.jpg: 736x960 (no detections), 210.4ms
image 3/780 D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\images\val2017\000000002157.jpg: 640x960 7 cups, 146.3ms
image 4/780 D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\images\val2017\000000002431.jpg: 960x704 4 cups, 266.3ms
image 5/780 D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\images\val2017\000000002592.jpg: 576x960 2 cups, 207.1ms
image 6/780 D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\images\val2017\000000002685.jpg: 832x960 1 cup, 229.2ms
image 7/780 D:\an4sem1\PRS\smart-object-detection\cup_coco_yolo_small\images\val2017\000000003553.jpg: 640x960 (no 

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'cup'}
 obb: None
 orig_img: array([[[ 14,  50,  20],
         [ 23,  54,  27],
         [ 54,  95,  68],
         ...,
         [ 31,  28,  20],
         [ 33,  20,  18],
         [ 31,  28,  20]],
 
        [[ 32,  63,  32],
         [ 21,  55,  25],
         [ 36,  80,  51],
         ...,
         [ 21,  26,  11],
         [ 33,  28,  19],
         [ 17,  17,   5]],
 
        [[ 42,  75,  41],
         [  6,  42,  12],
         [ 16,  60,  31],
         ...,
         [  9,  18,   0],
         [ 25,  25,  11],
         [ 21,  23,  11]],
 
        ...,
 
        [[177, 204, 224],
         [178, 205, 225],
         [177, 204, 225],
         ...,
         [175, 203, 227],
         [176, 204, 228],
         [175, 203, 227]],
 
        [[174, 201, 222],
         [175, 201, 225],
         [174, 200, 224],
         ...,
         [175, 203, 2

In [14]:
# Test prediction on a custom test folder

VAL_IMAGES_DIR = PROJECT_ROOT / "test_image"
print("Val images dir:", VAL_IMAGES_DIR, " | Exists:", VAL_IMAGES_DIR.exists())

sample_source = str(VAL_IMAGES_DIR)

pred_results = model.predict(
    source=sample_source,
    imgsz=960,
    conf = 0.15,
    iou = 0.55,
    save=True,
    project=str(OUTPUT_DIR),
    name=f"test_val_pred",
    exist_ok=True,
)

pred_results[:3]


Val images dir: D:\an4sem1\PRS\smart-object-detection\test_image  | Exists: True

image 1/3 D:\an4sem1\PRS\smart-object-detection\test_image\Image_created_with_a_mobile_phone.png: 736x960 (no detections), 229.1ms
image 2/3 D:\an4sem1\PRS\smart-object-detection\test_image\original.jpg: 736x960 5 cups, 204.0ms
image 3/3 D:\an4sem1\PRS\smart-object-detection\test_image\thumb.jpg: 960x736 4 cups, 158.1ms
Speed: 11.3ms preprocess, 197.0ms inference, 1.3ms postprocess per image at shape (1, 3, 960, 736)
Results saved to D:\an4sem1\PRS\smart-object-detection\outputs\test_val_pred


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'cup'}
 obb: None
 orig_img: array([[[249, 171,  95],
         [248, 170,  94],
         [246, 169,  96],
         ...,
         [210, 141,  78],
         [204, 137,  74],
         [204, 137,  74]],
 
        [[250, 172,  96],
         [249, 171,  95],
         [245, 170,  96],
         ...,
         [209, 141,  76],
         [207, 141,  76],
         [207, 141,  76]],
 
        [[248, 172,  96],
         [247, 171,  95],
         [245, 170,  96],
         ...,
         [210, 142,  77],
         [207, 142,  74],
         [209, 144,  76]],
 
        ...,
 
        [[ 60,  40,  35],
         [ 55,  36,  31],
         [ 50,  34,  28],
         ...,
         [  6,   4,   4],
         [  5,   3,   3],
         [  5,   3,   3]],
 
        [[ 54,  43,  35],
         [ 48,  39,  30],
         [ 45,  38,  29],
         ...,
         [  6,   4,  